In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# 1달치 목업 데이터 로드
df = pd.read_csv('output.csv')
print(f"원본 데이터 크기: {df.shape[0]}행 x {df.shape[1]}열")
df.head()

원본 데이터 크기: 30행 x 14열


,avg_dwell_time,core_customer_age,core_customer_gender,just_left_count,max_empty_table_time,max_response_wait_time,peak_time,temperature,total_count,captured_at,created_at,end_at,id,weather
0,70,30,1,0,18,4,19,19.4,52,2026-04-30 11:00:00.000000,2026-05-30 22:01:57.212084,2026-04-30 22:00:00.000000,1,SUNNY
1,67,30,1,1,12,7,18,19.8,35,2026-05-01 11:00:00.000000,2026-05-30 22:01:57.570298,2026-05-01 22:00:00.000000,2,SUNNY
2,68,30,1,1,32,4,19,18.5,59,2026-05-02 11:00:00.000000,2026-05-30 22:01:57.842495,2026-05-02 22:00:00.000000,3,CLOUDY
3,68,30,1,0,21,5,19,13.2,64,2026-05-03 11:00:00.000000,2026-05-30 22:02:01.373380,2026-05-03 22:00:00.000000,4,RAINY
4,72,30,1,3,19,4,18,16.1,37,2026-05-04 11:00:00.000000,2026-05-30 22:02:01.782206,2026-05-04 22:00:00.000000,5,CLOUDY


In [3]:
# [피처 엔지니어링 1단계] captured_at 날짜 파싱 및 요일 번호 생성
df['captured_at'] = pd.to_datetime(df['captured_at'])
df['day_of_week'] = df['captured_at'].dt.dayofweek
print("요일(day_of_week: 월=0 ~ 일=6) 피처 생성 완료:")
df[['captured_at', 'day_of_week']].head()

요일(day_of_week: 월=0 ~ 일=6) 피처 생성 완료:


,captured_at,day_of_week
0,2026-04-30 11:00:00,3
1,2026-05-01 11:00:00,4
2,2026-05-02 11:00:00,5
3,2026-05-03 11:00:00,6
4,2026-05-04 11:00:00,0


In [4]:
# [피처 엔지니어링 2단계] 주말 여부(is_weekend) 바이너리 변수 생성
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
print("주말 여부(is_weekend) 피처 생성 완료:")
df[['captured_at', 'day_of_week', 'is_weekend']].head()

주말 여부(is_weekend) 피처 생성 완료:


,captured_at,day_of_week,is_weekend
0,2026-04-30 11:00:00,3,0
1,2026-05-01 11:00:00,4,0
2,2026-05-02 11:00:00,5,1
3,2026-05-03 11:00:00,6,1
4,2026-05-04 11:00:00,0,0


In [5]:
# [피처 엔지니어링 3단계] 어제 방문객 수(prev_day_count) 시계열 지연 생성 및 결측치 제거
df['prev_day_count'] = df['total_count'].shift(1)

# 첫째 날은 전날 카운트가 없어 NaN이 생기므로 삭제 처리
df_clean = df.dropna().reset_index(drop=True)
print(f"지연 피처 생성 및 결측치 제거 완료 (잔여 데이터: {df_clean.shape[0]}개):")
df_clean[['captured_at', 'is_weekend', 'prev_day_count', 'total_count']].head()

지연 피처 생성 및 결측치 제거 완료 (잔여 데이터: 29개):


,captured_at,is_weekend,prev_day_count,total_count
0,2026-05-01 11:00:00,0,52.0,35
1,2026-05-02 11:00:00,1,35.0,59
2,2026-05-03 11:00:00,1,59.0,64
3,2026-05-04 11:00:00,0,64.0,37
4,2026-05-05 11:00:00,0,37.0,38


In [6]:
# [피처 엔지니어링 4단계] 범주형 기상(weather) 데이터 원-핫 인코딩 적용
df_encoded = pd.get_dummies(df_clean, columns=['weather'], drop_first=True)
print("날씨 범주형 변수 원핫 인코딩(Dummy 변수화) 완료:")
df_encoded.head()

날씨 범주형 변수 원핫 인코딩(Dummy 변수화) 완료:


,avg_dwell_time,core_customer_age,core_customer_gender,just_left_count,max_empty_table_time,max_response_wait_time,peak_time,temperature,total_count,captured_at,created_at,end_at,id,day_of_week,is_weekend,prev_day_count,weather_RAINY,weather_SUNNY
0,67,30,1,1,12,7,18,19.8,35,2026-05-01 11:00:00,2026-05-30 22:01:57.570298,2026-05-01 22:00:00.000000,2,4,0,52.0,False,True
1,68,30,1,1,32,4,19,18.5,59,2026-05-02 11:00:00,2026-05-30 22:01:57.842495,2026-05-02 22:00:00.000000,3,5,1,35.0,False,False
2,68,30,1,0,21,5,19,13.2,64,2026-05-03 11:00:00,2026-05-30 22:02:01.373380,2026-05-03 22:00:00.000000,4,6,1,59.0,True,False
3,72,30,1,3,19,4,18,16.1,37,2026-05-04 11:00:00,2026-05-30 22:02:01.782206,2026-05-04 22:00:00.000000,5,0,0,64.0,False,False
4,67,30,1,1,32,4,18,16.9,38,2026-05-05 11:00:00,2026-05-30 22:02:02.046001,2026-05-05 22:00:00.000000,6,1,0,37.0,False,True


In [7]:
# [피처 엔지니어링 5단계] 미래 정보 배제(Data Leakage 차단) 및 최종 피처 선택
features = ['temperature', 'is_weekend', 'prev_day_count']
features += [col for col in df_encoded.columns if col.startswith('weather_')]

X = df_encoded[features]
y = df_encoded['total_count']

print("최종 선정된 릿지 모델 입력 피처(X) 구조:")
X.head()

최종 선정된 릿지 모델 입력 피처(X) 구조:


,temperature,is_weekend,prev_day_count,weather_RAINY,weather_SUNNY
0,19.8,0,52.0,False,True
1,18.5,1,35.0,False,False
2,13.2,1,59.0,True,False
3,16.1,0,64.0,False,False
4,16.9,0,37.0,False,True


In [8]:
# 릿지 회귀 규제 작동을 위한 StandardScaler 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 학습용 80%, 검증용 20% 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"학습용 샘플 수: {X_train.shape[0]}개 | 검증용 샘플 수: {X_test.shape[0]}개")
print("\n첫 5개 행 스케일링 적용 후 numpy 배열:")
X_scaled[:5]

학습용 샘플 수: 23개 | 검증용 샘플 수: 6개

첫 5개 행 스케일링 적용 후 numpy 배열:


array([[-0.37238664, -0.6172134 ,  0.36677296, -0.45643546,  1.37840488],
       [-0.7050638 ,  1.62018517, -1.17868918, -0.45643546, -0.72547625],
       [-2.061363  ,  1.62018517,  1.00313973,  2.19089023, -0.72547625],
       [-1.31923702, -0.6172134 ,  1.45768741, -0.45643546, -0.72547625],
       [-1.11451262, -0.6172134 , -0.9968701 , -0.45643546,  1.37840488]])

In [9]:
# 릿지 회귀 모델 객체 선언 및 학습
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# 검증 평가 및 성능 출력
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=================== 📊 모델 평가 결과 ===================")
print(f"평균 절대 오차 (MAE): {mae:.2f} 명")
print(f"결정계수 (R² Score): {r2:.4f}")
print("==========================================================\n")

print("=================== 🔑 피처별 가중치 (Coefficients) ===================")
for feat, coef in zip(features, model.coef_):
    print(f" - {feat}: {coef:+.4f}")

=================== 📊 모델 평가 결과 ===================
평균 절대 오차 (MAE): 5.87 명
결정계수 (R² Score): 0.6841

=================== 🔑 피처별 가중치 (Coefficients) ===================
 - temperature: +0.8715
 - is_weekend: +8.3198
 - prev_day_count: -0.7832
 - weather_RAINY: +1.3463
 - weather_SUNNY: -0.1949


In [10]:
def predict_tomorrow(tomorrow_temp, tomorrow_weather, today_count, model, scaler, feature_columns):
    # 내일 날짜 요일 기준 계산 (오늘 + 1일)
    tomorrow_date = pd.Timestamp.now() + pd.Timedelta(days=1)
    day_of_week = tomorrow_date.dayofweek
    is_weekend = 1 if day_of_week >= 5 else 0
    
    input_data = {
        'temperature': tomorrow_temp,
        'is_weekend': is_weekend,
        'prev_day_count': today_count
    }
    
    # 원핫 인코딩 날씨 매핑
    for col in feature_columns:
        if col.startswith('weather_'):
            weather_type = col.replace('weather_', '')
            input_data[col] = 1 if tomorrow_weather.upper() == weather_type.upper() else 0
            
    input_df = pd.DataFrame([input_data])[feature_columns]
    input_scaled = scaler.transform(input_df)
    
    # 예측 수행
    pred_raw = model.predict(input_scaled)[0]
    final_pred = max(0, int(round(pred_raw)))
    
    kor_days = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    
    print(f"🔮 [SPOTLINE AI 내일 예측 보고서]")
    print(f"  - 예측 기준 일자: {tomorrow_date.strftime('%Y-%m-%d')} ({kor_days[day_of_week]})")
    print(f"  - 내일 최고 기온: {tomorrow_temp} ℃")
    print(f"  - 내일 기상 상태: {tomorrow_weather}")
    print(f"  - 오늘 최종 방문자수: {today_count} 명")
    print(f"--------------------------------------------------")
    print(f"👉 내일 삼겹살집 예상 방문객 수: 【 {final_pred} 명 】")
    return final_pred

# 가상 상황 테스트 실행
print("[가상 시뮬레이션 가동]")
predict_tomorrow(
    tomorrow_temp=21.5, 
    tomorrow_weather='RAINY', 
    today_count=42, 
    model=model, 
    scaler=scaler, 
    feature_columns=features
)

[가상 시뮬레이션 가동]
🔮 [SPOTLINE AI 내일 예측 보고서]
  - 예측 기준 일자: 2026-06-01 (월요일)
  - 내일 최고 기온: 21.5 ℃
  - 내일 기상 상태: RAINY
  - 오늘 최종 방문자수: 42 명
--------------------------------------------------
👉 내일 삼겹살집 예상 방문객 수: 【 46 명 】


46